# PGA Tour Hot Hand Study — Part 1: Data Collection

This notebook pulls five seasons (2020–2024) of PGA Tour data from ESPN's public API. No API key needed — these are the same endpoints powering the ESPN app.

**Three things we collect:**
- **Schedules** — every event on the PGA Tour calendar per year
- **Leaderboards** — final standings for every tournament
- **Hole-by-hole scoring** — the main dataset, pulled for marquee events (roughly 15/year) and the top 30 finishers in each

**Output files saved to `data/`:**
```
schedule_2020_2024.parquet
leaderboards_2020_2024.parquet
holes_2020_2024.parquet
```

**Expected runtime:** 25–40 minutes (we sleep between requests to avoid hammering the API)


In [4]:
import requests
import pandas as pd
import numpy as np
import time
import os

os.makedirs("data", exist_ok=True)
print("Setup complete.")

Setup complete.


In [5]:
# API endpoints — no key required
ESPN_SITE = "https://site.api.espn.com/apis/site/v2/sports/golf/pga/"
ESPN_CORE = "https://sports.core.api.espn.com/v2/sports/golf/leagues/pga/"
HEADERS   = {"Accept": "application/json"}

YEARS = [2020, 2021, 2022, 2023, 2024]
SLEEP = 0.45  # seconds between requests

MARQUEE_KEYWORDS = [
    "masters", "pga championship", "u.s. open", "open championship",
    "the players", "arnold palmer", "genesis", "rbc heritage",
    "memorial", "travelers", "tour championship", "bmw championship",
    "northern trust", "fedex st. jude", "scottish open"
]


def get_json(url, params=None):
    r = requests.get(url, headers=HEADERS, params=params, timeout=15)
    r.raise_for_status()
    return r.json()


def safe_get(url, params=None, tag=""):
    """Like get_json but swallows errors and returns None on failure."""
    try:
        return get_json(url, params)
    except Exception as e:
        print(f"    skip [{tag}] — {e}")
        return None


def is_marquee(name):
    n = name.lower()
    return any(k in n for k in MARQUEE_KEYWORDS)


print("Config loaded.")

Config loaded.


## 1. Season Schedules

Pull the calendar for each year — event IDs, names, and dates. We need the IDs to query everything else.

In [6]:
def fetch_schedule(year):
    data = get_json(ESPN_SITE + "scoreboard", params={"dates": year})
    events = data["leagues"][0]["calendar"]
    return pd.DataFrame({
        "event_id":   [str(e["id"])       for e in events],
        "name":       [e["label"]         for e in events],
        "start_date": [e["startDate"][:10] for e in events],
        "end_date":   [e["endDate"][:10]   for e in events],
        "year":       year,
    })


print("Fetching schedules for 2020–2024...")
all_sched = []

for yr in YEARS:
    sched = fetch_schedule(yr)
    all_sched.append(sched)
    print(f"  {yr}: {len(sched)} events")
    time.sleep(SLEEP)

schedule = pd.concat(all_sched, ignore_index=True)
schedule.to_parquet("data/schedule_2020_2024.parquet", index=False)
print(f"\nTotal: {len(schedule)} tournaments | saved to data/schedule_2020_2024.parquet")

Fetching schedules for 2020–2024...
  2020: 50 events
  2021: 55 events
  2022: 51 events
  2023: 61 events
  2024: 51 events

Total: 268 tournaments | saved to data/schedule_2020_2024.parquet


## 2. Leaderboards

For each tournament in the schedule, pull the full field results — finishing position, player name, total score, and how many rounds they completed. This feeds both the performance benchmark in the analysis and lets us know which players to pull hole data for.

In [7]:
def parse_leaderboard(event_id):
    data = safe_get(ESPN_SITE + f"scoreboard/{event_id}", tag=event_id)
    if not data:
        return None

    comps = data.get("competitions", [])
    if not comps:
        return None

    rows = []
    for c in comps[0].get("competitors", []):
        rnd_scores = [
            ls["value"] for ls in c.get("linescores", [])
            if ls.get("period") in [1, 2, 3, 4] and ls.get("value") is not None
        ]
        rows.append({
            "position":      c.get("order"),
            "player_id":     str(c.get("id", "")),
            "player_name":   c.get("athlete", {}).get("displayName", ""),
            "score_to_par":  c.get("score", ""),
            "total_strokes": sum(rnd_scores) if rnd_scores else None,
            "rounds_played": len(rnd_scores),
            "status":        c.get("status", {}).get("type", {}).get("description", ""),
        })
    return pd.DataFrame(rows) if rows else None


print(f"Fetching leaderboards for {len(schedule)} tournaments...")
print("(This takes 3-5 minutes)\n")

all_lb = []
for i, row in schedule.iterrows():
    lb = parse_leaderboard(row["event_id"])
    if lb is not None and not lb.empty:
        lb["tournament_id"]   = row["event_id"]
        lb["tournament_name"] = row["name"]
        lb["year"]            = row["year"]
        all_lb.append(lb)
    if (i + 1) % 40 == 0:
        pct = (i + 1) / len(schedule) * 100
        print(f"  {i+1}/{len(schedule)} ({pct:.0f}%)")
    time.sleep(SLEEP)

leaderboards = pd.concat(all_lb, ignore_index=True)
leaderboards.to_parquet("data/leaderboards_2020_2024.parquet", index=False)
print(f"\nSaved {len(leaderboards):,} player-tournament rows")

Fetching leaderboards for 268 tournaments...
(This takes 3-5 minutes)

  40/268 (15%)
    skip [401243007] — 500 Server Error: Internal Server Error for url: https://site.api.espn.com/apis/site/v2/sports/golf/pga/scoreboard/401243007
  80/268 (30%)
  120/268 (45%)
    skip [401353293] — 500 Server Error: Internal Server Error for url: https://site.api.espn.com/apis/site/v2/sports/golf/pga/scoreboard/401353293
  160/268 (60%)
    skip [401465528] — 500 Server Error: Internal Server Error for url: https://site.api.espn.com/apis/site/v2/sports/golf/pga/scoreboard/401465528
  200/268 (75%)
  240/268 (90%)

Saved 29,377 player-tournament rows


/tmp/ipykernel_406/1616178792.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  leaderboards = pd.concat(all_lb, ignore_index=True)


## 3. Hole-by-Hole Scoring

This is the core dataset. For each marquee tournament, we pull hole-by-hole scores for the top 30 finishers.

Why top 30? It's a pragmatic tradeoff — pulling all 70+ finishers would take hours and adds mostly players who didn't make the cut. The top 30 covers a solid range of skill within the field.

Why marquee events only? ESPN's hole-level data is most complete and reliable for these events.

In [8]:
def fetch_holes_for_event(event_id, lb_df, top_n=30):
    """Pull hole-by-hole data for top_n players in a given tournament."""
    player_ids = lb_df.head(top_n)["player_id"].tolist()
    rows = []

    for pid in player_ids:
        url = (
            ESPN_CORE
            + f"events/{event_id}/competitions/{event_id}"
            + f"/competitors/{pid}/linescores"
        )
        data = safe_get(url, tag=f"{event_id}/{pid}")
        if not data or not data.get("items"):
            continue

        names = lb_df.loc[lb_df["player_id"] == pid, "player_name"].values
        pname = names[0] if len(names) else "Unknown"

        for item in data["items"]:
            rnd = item.get("period")
            for ls in item.get("linescores", []):
                score = ls.get("value")
                par   = ls.get("par")
                if score is None or par is None:
                    continue
                rows.append({
                    "player_id":    pid,
                    "player_name":  pname,
                    "round":        rnd,
                    "hole":         ls.get("period"),
                    "par":          par,
                    "score":        score,
                    "score_to_par": score - par,
                    "score_type":   (ls.get("scoreType") or {}).get("name", ""),
                })
        time.sleep(0.3)

    return pd.DataFrame(rows) if rows else None


# Filter to marquee events
target = schedule[schedule["name"].apply(is_marquee)].copy()
target = target.groupby("year").head(15).reset_index(drop=True)

print(f"Target: {len(target)} events across 5 years")
print("\nSelected tournaments:")
for _, row in target.iterrows():
    print(f"  {row['year']}  {row['name']}")

Target: 66 events across 5 years

Selected tournaments:
  2020  The Genesis Invitational
  2020  Arnold Palmer Invitational Pres. By Mastercard
  2020  THE PLAYERS Championship
  2020  RBC Heritage
  2020  Travelers Championship
  2020  The Open Championship
  2020  the Memorial Tournament pres. by Nationwide
  2020  WGC-FedEx St. Jude Invitational
  2020  PGA Championship
  2020  THE NORTHERN TRUST
  2020  BMW Championship
  2020  Tour Championship
  2021  U.S. Open
  2021  2020 Masters Tournament
  2021  The Genesis Invitational
  2021  Arnold Palmer Invitational Pres. By Mastercard
  2021  THE PLAYERS Championship
  2021  2021 Masters Tournament
  2021  RBC Heritage
  2021  PGA Championship
  2021  the Memorial Tournament pres. by Nationwide
  2021  U.S. Open
  2021  Travelers Championship
  2021  The Open Championship
  2021  WGC-FedEx St. Jude Invitational
  2021  THE NORTHERN TRUST
  2021  BMW Championship
  2022  The Genesis Invitational
  2022  Arnold Palmer Invitational pres. 

In [9]:
print("\nFetching hole-by-hole data...")
print("(This takes 15-25 minutes)\n")

all_holes = []
for i, row in target.iterrows():
    # Get leaderboard for this specific event
    event_lb = leaderboards[leaderboards["tournament_id"] == row["event_id"]]
    if event_lb.empty:
        print(f"  [{i+1}] no leaderboard for {row['name']} — skipping")
        continue

    print(f"  [{i+1}/{len(target)}] {row['year']} {row['name']}")
    holes = fetch_holes_for_event(row["event_id"], event_lb, top_n=30)

    if holes is not None and not holes.empty:
        holes["tournament_id"]   = row["event_id"]
        holes["tournament_name"] = row["name"]
        holes["year"]            = row["year"]
        all_holes.append(holes)

    time.sleep(SLEEP)

holes_df = pd.concat(all_holes, ignore_index=True)
holes_df.to_parquet("data/holes_2020_2024.parquet", index=False)
print(f"\nSaved {len(holes_df):,} hole-level rows")


Fetching hole-by-hole data...
(This takes 15-25 minutes)

  [1/66] 2020 The Genesis Invitational
  [2/66] 2020 Arnold Palmer Invitational Pres. By Mastercard
  [3] no leaderboard for THE PLAYERS Championship — skipping
  [4/66] 2020 RBC Heritage
  [5/66] 2020 Travelers Championship
  [6] no leaderboard for The Open Championship — skipping
  [7/66] 2020 the Memorial Tournament pres. by Nationwide
  [8/66] 2020 WGC-FedEx St. Jude Invitational
  [9/66] 2020 PGA Championship
  [10/66] 2020 THE NORTHERN TRUST
  [11/66] 2020 BMW Championship
  [12/66] 2020 Tour Championship
  [13/66] 2021 U.S. Open
  [14/66] 2021 2020 Masters Tournament
  [15/66] 2021 The Genesis Invitational
  [16/66] 2021 Arnold Palmer Invitational Pres. By Mastercard
  [17/66] 2021 THE PLAYERS Championship
  [18/66] 2021 2021 Masters Tournament
  [19/66] 2021 RBC Heritage
  [20/66] 2021 PGA Championship
  [21/66] 2021 the Memorial Tournament pres. by Nationwide
  [22/66] 2021 U.S. Open
  [23/66] 2021 Travelers Championsh

## 4. Quick Sanity Check

In [10]:
schedule    = pd.read_parquet("data/schedule_2020_2024.parquet")
leaderboards = pd.read_parquet("data/leaderboards_2020_2024.parquet")
holes_df     = pd.read_parquet("data/holes_2020_2024.parquet")

print("=" * 55)
print("DATA COLLECTION SUMMARY")
print("=" * 55)
print(f"  Schedule:       {len(schedule):>7,} tournaments")
print(f"  Leaderboards:   {len(leaderboards):>7,} player-tournament rows")
print(f"  Holes:          {len(holes_df):>7,} hole-level observations")
print()
print("Hole data by year:")
print(holes_df.groupby("year")["hole"].count().rename("holes").to_string())
print()
print("Hole data by tournament:")
print(holes_df.groupby(["year", "tournament_name"])["hole"].count()
      .rename("holes").reset_index().to_string(index=False))
print()
print("Score types observed:")
print(holes_df["score_type"].value_counts().head(10))

DATA COLLECTION SUMMARY
  Schedule:           268 tournaments
  Leaderboards:    29,377 player-tournament rows
  Holes:          138,229 hole-level observations

Hole data by year:
year
2020    21602
2021    32439
2022    28019
2023    28088
2024    28081

Hole data by tournament:
 year                                tournament_name  holes
 2020 Arnold Palmer Invitational Pres. By Mastercard   2160
 2020                               BMW Championship   2162
 2020                               PGA Championship   2160
 2020                                   RBC Heritage   2160
 2020                             THE NORTHERN TRUST   2160
 2020                       The Genesis Invitational   2160
 2020                              Tour Championship   2160
 2020                         Travelers Championship   2160
 2020                WGC-FedEx St. Jude Invitational   2160
 2020    the Memorial Tournament pres. by Nationwide   2160
 2021                        2020 Masters Tournament   216